# 1) ENVIRONMENT SETUP

In [1]:
%pip install --upgrade torch torchvision torchaudio
%pip install --upgrade vllm
%pip install transformers
%pip install accelerate
%pip install bitsandbytes
%pip install fastapi uvicorn
%pip install gradio
%pip install tqdm
%pip install -U ipywidgets


  Using cached torch-2.10.0-cp312-cp312-manylinux_2_28_x86_64.whl.metadata (31 kB)
  Using cached torchvision-0.25.0-cp312-cp312-manylinux_2_28_x86_64.whl.metadata (5.4 kB)
  Using cached torchaudio-2.10.0-cp312-cp312-manylinux_2_28_x86_64.whl.metadata (6.9 kB)
  Using cached cuda_bindings-12.9.4-cp312-cp312-manylinux_2_24_x86_64.manylinux_2_28_x86_64.whl.metadata (2.6 kB)
  Using cached nvidia_nvshmem_cu12-3.4.5-py3-none-manylinux2014_x86_64.manylinux_2_17_x86_64.whl.metadata (2.1 kB)
  Using cached triton-3.6.0-cp312-cp312-manylinux_2_27_x86_64.manylinux_2_28_x86_64.whl.metadata (1.7 kB)
Using cached torch-2.10.0-cp312-cp312-manylinux_2_28_x86_64.whl (915.7 MB)
Using cached cuda_bindings-12.9.4-cp312-cp312-manylinux_2_24_x86_64.manylinux_2_28_x86_64.whl (12.2 MB)
Using cached nvidia_nvshmem_cu12-3.4.5-py3-none-manylinux2014_x86_64.manylinux_2_17_x86_64.whl (139.1 MB)
Using cached triton-3.6.0-cp312-cp312-manylinux_2_27_x86_64.manylinux_2_28_x86_64.whl (188.3 MB)
Using cached torchvis

In [2]:
import torch
print(torch.cuda.is_available())
torch.version.cuda
print(torch.cuda.get_device_name(0))

True
NVIDIA H100 PCIe


# 2) HUGGINGFACE INFERENCE (BASELINE)

In [3]:
from tqdm import tqdm
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch
import time

tqdm.pandas()

model_name = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"

tokenizer = AutoTokenizer.from_pretrained(model_name)

hf_model = AutoModelForCausalLM.from_pretrained(
    model_name,
    dtype=torch.float16,
    device_map="auto"
)

def hf_generate(prompt, max_new_tokens=100):
    messages = [
        {"role": "system", "content": "You are a helpful assistant."},
        {"role": "user", "content": prompt}
    ]

    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    inputs = tokenizer(text, return_tensors="pt").to("cuda")

    torch.cuda.synchronize()
    start = time.perf_counter()

    outputs = hf_model.generate(
        **inputs,
        max_new_tokens=max_new_tokens
    )

    torch.cuda.synchronize()
    end = time.perf_counter()

    new_tokens = outputs[0][inputs["input_ids"].shape[-1]:]
    generated_text = tokenizer.decode(new_tokens, skip_special_tokens=True)

    latency = end - start
    tokens_generated = len(new_tokens)

    return latency, tokens_generated, generated_text

# 3) vLLM INFERENCE (BENCHMARK)

In [4]:
import os
import time # Ensure time is imported for your latency check
from vllm import LLM, SamplingParams

# Force V0 and disable the experimental V1 components entirely
os.environ["VLLM_USE_V1"] = "0"
os.environ["VLLM_V1_INPROC"] = "0"

# Initialize LLM for single-GPU
llm = LLM(
    model=model_name,
    dtype="float16",
    trust_remote_code=True, # Often needed for modern models 
    enforce_eager=True,          # Bypasses complex CUDA graph capture
    disable_log_stats=True,   # <--- This stops the periodic widget updates
    gpu_memory_utilization=0.8, # Leave room for the system
    tensor_parallel_size=1       # Ensure single-process
)



sampling_params = SamplingParams(
    temperature=0.7,
    max_tokens=100
)

def vllm_generate(prompt):
    messages = [
        {"role": "system", "content": "You are a helpful assistant."},
        {"role": "user", "content": prompt}
    ]

    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    start = time.perf_counter()
    outputs = llm.generate([text], sampling_params, use_tqdm=False)
    end = time.perf_counter()

    latency = end - start
    generated_text = outputs[0].outputs[0].text
    tokens_generated = len(tokenizer(generated_text)["input_ids"])

    return latency, tokens_generated, generated_text

INFO 03-02 00:43:48 [utils.py:223] non-default args: {'trust_remote_code': True, 'dtype': 'float16', 'gpu_memory_utilization': 0.8, 'disable_log_stats': True, 'enforce_eager': True, 'model': 'TinyLlama/TinyLlama-1.1B-Chat-v1.0'}


The argument `trust_remote_code` is to be used with Auto classes. It has no effect here and is ignored.
The argument `trust_remote_code` is to be used with Auto classes. It has no effect here and is ignored.


INFO 03-02 00:43:48 [model.py:529] Resolved architecture: LlamaForCausalLM
WARNING 03-02 00:43:49 [model.py:1874] Casting torch.bfloat16 to torch.float16.
INFO 03-02 00:43:49 [model.py:1549] Using max model len 2048
INFO 03-02 00:43:49 [scheduler.py:224] Chunked prefill is enabled with max_num_batched_tokens=16384.
INFO 03-02 00:43:49 [vllm.py:689] Asynchronous scheduling is enabled.
WARNING 03-02 00:43:49 [vllm.py:727] Enforce eager set, overriding optimization level to -O0
INFO 03-02 00:43:49 [vllm.py:845] Cudagraph is disabled under eager mode
WARNING 03-02 00:43:49 [system_utils.py:140] We must use the `spawn` multiprocessing start method. Overriding VLLM_WORKER_MULTIPROC_METHOD to 'spawn'. See https://docs.vllm.ai/en/latest/usage/troubleshooting.html#python-multiprocessing for more information. Reasons: CUDA is initialized
(EngineCore_DP0 pid=6196) INFO 03-02 00:43:55 [core.py:97] Initializing a V1 LLM engine (v0.16.0) with config: model='TinyLlama/TinyLlama-1.1B-Chat-v1.0', specu

[rank0]:[W302 00:43:58.616537312 ProcessGroupGloo.cpp:516] Warning: Unable to resolve hostname to a (local) address. Using the loopback address as fallback. Manually set the network interface to bind to with GLOO_SOCKET_IFNAME. (function operator())


(EngineCore_DP0 pid=6196) INFO 03-02 00:43:58 [gpu_model_runner.py:4124] Starting to load model TinyLlama/TinyLlama-1.1B-Chat-v1.0...
(EngineCore_DP0 pid=6196) INFO 03-02 00:43:59 [cuda.py:367] Using FLASH_ATTN attention backend out of potential backends: ['FLASH_ATTN', 'FLASHINFER', 'TRITON_ATTN', 'FLEX_ATTENTION'].
(EngineCore_DP0 pid=6196) INFO 03-02 00:44:00 [weight_utils.py:579] No model.safetensors.index.json found in remote.


Loading safetensors checkpoint shards:   0% Completed | 0/1 [00:00<?, ?it/s]
Loading safetensors checkpoint shards: 100% Completed | 1/1 [00:01<00:00,  1.75s/it]
Loading safetensors checkpoint shards: 100% Completed | 1/1 [00:01<00:00,  1.75s/it]
(EngineCore_DP0 pid=6196) 


(EngineCore_DP0 pid=6196) INFO 03-02 00:44:01 [default_loader.py:293] Loading weights took 1.87 seconds
(EngineCore_DP0 pid=6196) INFO 03-02 00:44:02 [gpu_model_runner.py:4221] Model loading took 2.05 GiB memory and 3.033392 seconds
(EngineCore_DP0 pid=6196) INFO 03-02 00:44:04 [gpu_worker.py:373] Available KV cache memory: 57.81 GiB
(EngineCore_DP0 pid=6196) INFO 03-02 00:44:04 [kv_cache_utils.py:1307] GPU KV cache size: 2,755,216 tokens
(EngineCore_DP0 pid=6196) INFO 03-02 00:44:04 [kv_cache_utils.py:1312] Maximum concurrency for 2,048 tokens per request: 1345.32x
(EngineCore_DP0 pid=6196) INFO 03-02 00:44:04 [kernel_warmup.py:44] Skipping FlashInfer autotune because it is disabled.
(EngineCore_DP0 pid=6196) INFO 03-02 00:44:04 [core.py:278] init engine (profile, create kv cache, warmup model) took 1.84 seconds
(EngineCore_DP0 pid=6196) INFO 03-02 00:44:04 [vllm.py:689] Asynchronous scheduling is enabled.
(EngineCore_DP0 pid=6196) WARNING 03-02 00:44:04 [vllm.py:734] Inductor compila

# 4) Single Prompt Fair Comparison

In [5]:
prompt = "Explain KV cache in simple words."

hf_latency, hf_tokens, hf_output = hf_generate(prompt)
vllm_latency, vllm_tokens, vllm_output = vllm_generate(prompt)

print("---- SINGLE REQUEST ----")
print(f"HF Latency: {hf_latency:.4f}s")
print(f"vLLM Latency: {vllm_latency:.4f}s")

print(f"HF Tokens/sec: {hf_tokens/hf_latency:.2f}")
print(f"vLLM Tokens/sec: {vllm_tokens/vllm_latency:.2f}")

---- SINGLE REQUEST ----
HF Latency: 3.7202s
vLLM Latency: 5.6765s
HF Tokens/sec: 26.88
vLLM Tokens/sec: 17.79


# 5) Average Over Multiple Prompts (More Realistic Scenarios)

In [6]:
test_prompts = [
    "Explain transformers in simple words.",
    "What is gradient descent?",
    "Explain overfitting in ML.",
    "What is KV cache?",
    "Describe neural networks."
]

hf_latencies = []
vllm_latencies = []

for p in test_prompts:
    hf_lat, _, _ = hf_generate(p)
    vllm_lat, _, _ = vllm_generate(p)

    hf_latencies.append(hf_lat)
    vllm_latencies.append(vllm_lat)

print("---- AVERAGE LATENCY ----")
print("HF Avg Latency:", sum(hf_latencies)/len(hf_latencies))
print("vLLM Avg Latency:", sum(vllm_latencies)/len(vllm_latencies))

---- AVERAGE LATENCY ----
HF Avg Latency: 2.102921123057604
vLLM Avg Latency: 5.714391223248095


# 6) TOKENS/SEC BENCHMARK

In [8]:
hf_tps = []
vllm_tps = []

for p in test_prompts:
    hf_lat, hf_tok, _ = hf_generate(p)
    vllm_lat, vllm_tok, _ = vllm_generate(p)

    hf_tps.append(hf_tok/hf_lat)
    vllm_tps.append(vllm_tok/vllm_lat)

print("TOKENS PER SECOND: ")
print("HF Avg Tokens/sec:", sum(hf_tps)/len(hf_tps))
print("vLLM Avg Tokens/sec:", sum(vllm_tps)/len(vllm_tps))

TOKENS PER SECOND: 
HF Avg Tokens/sec: 46.409879895545636
vLLM Avg Tokens/sec: 18.15737889348871


# 7) BATCH THROUGHPUT SCALING

In [10]:
batch_sizes = [1, 4, 8, 16, 32]
results = []

for batch in batch_sizes:
    prompts = ["Explain transformers in simple words."] * batch

    # HF Sequential
    start = time.perf_counter()
    for p in prompts:
        hf_generate(p)
    end = time.perf_counter()
    hf_throughput = batch / (end - start)

    # vLLM Batch
    texts = []
    for p in prompts:
        messages = [
            {"role": "system", "content": "You are a helpful assistant."},
            {"role": "user", "content": p}
        ]
        text = tokenizer.apply_chat_template(
            messages,
            tokenize=False,
            add_generation_prompt=True
        )
        texts.append(text)

    start = time.perf_counter()
    llm.generate(texts, sampling_params, use_tqdm=False)
    end = time.perf_counter()
    vllm_throughput = batch / (end - start)

    results.append((batch, hf_throughput, vllm_throughput))

print("THROUGHPUT RESULTS:")
for r in results:
    print(f"Batch {r[0]} | HF: {r[1]:.2f} req/s | vLLM: {r[2]:.2f} req/s")

THROUGHPUT RESULTS:
Batch 1 | HF: 0.44 req/s | vLLM: 0.18 req/s
Batch 4 | HF: 0.43 req/s | vLLM: 0.70 req/s
Batch 8 | HF: 0.44 req/s | vLLM: 1.36 req/s
Batch 16 | HF: 0.43 req/s | vLLM: 2.77 req/s
Batch 32 | HF: 0.45 req/s | vLLM: 5.48 req/s
